# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [8]:
import os

print(os.listdir(DATA_ROOT))

['ESC-50-master', 'sample_submission.csv', 'genres_stems', 'mashups', 'test.csv']


In [9]:
print(os.listdir(os.path.join(DATA_ROOT, "genres_stems")))

['disco', 'metal', 'reggae', 'blues', 'rock', 'classical', 'jazz', 'hiphop', 'country', 'pop']


In [10]:
rock_path = os.path.join(DATA_ROOT, "genres_stems", "rock")
print(os.listdir(rock_path)[:5])

['rock.00052', 'rock.00083', 'rock.00092', 'rock.00053', 'rock.00096']


In [11]:
song_path = os.path.join(DATA_ROOT, "genres_stems", "rock", os.listdir(rock_path)[0])
print(os.listdir(song_path))

['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']


In [12]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"  # Enter dataset path
GENRES = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]  # Make the list of all genres available (alphabetical order)
STEMS = {"drums": "drums.wav",
    "vocals": "vocals.wav",
    "bass": "bass.wav",
    "other": "other.wav"}    # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [13]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

    import os

    for genre in GENRES:
        genre_path = os.path.join(root_dir, "genres_stems", genre)
    
        # Skip if folder missing
        if not os.path.isdir(genre_path):
            continue
    
        songs = sorted(os.listdir(genre_path))
        valid_songs = []
    
        # --- check each song folder ---
        for song in songs:
            song_path = os.path.join(genre_path, song)
    
            if not os.path.isdir(song_path):
                continue
    
            stem_paths = {}
            corrupted = False
    
            # Check stems exist and are not corrupted
            for key in STEM_KEYS:
                file_name = STEMS[key]
                file_path = os.path.join(song_path, file_name)
    
                if not os.path.isfile(file_path):
                    corrupted = True
                    break
    
                # check file size (corruption test)
                if os.path.getsize(file_path) < 4 * 1024:
                    corrupted = True
                    break
    
                stem_paths[key] = file_path
    
            if not corrupted:
                valid_songs.append(stem_paths)
    
        # --- shuffle songs reproducibly ---
        rng.shuffle(valid_songs)
    
        # --- split into train / validation ---
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]
    
        # --- helper function to populate dict ---
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem_key, path in song.items():
                    target_dict[genre][stem_key].append(path)
    
        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    
        # Iterate through Genres
        # Check: if genre folder exists
        # CHECK : Completeness (Does it have all stems?)
        # CHECK : Corruption (Is any file too small? (less than 4kb))
        # size checks
        # Stratified Shuffle Split
     #-------------------------------------------------------------------------

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            pass

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

In [14]:
print(len(tr["rock"]["drums"]))
print(len(val["rock"]["drums"]))

83
17


In [26]:
# Q1

small_4kb = 0
small_5mb = 0

for genre in GENRES:
    genre_dir = os.path.join(DATA_ROOT, "genres_stems", genre)
    songs = os.listdir(genre_dir)

    for song in songs:
        song_dir = os.path.join(genre_dir, song)

        for stem in STEMS.values():
            file_path = os.path.join(song_dir, stem)
            if not os.path.isfile(file_path):
                continue

            size = os.path.getsize(file_path)

            if size < 4 * 1024:
                small_4kb += 1
            if size < 5.0491 * 1024 * 1024:
                small_5mb += 1

print("Q1:", small_4kb + small_5mb)

Q1: 1256


In [27]:
# Q2

large = 0
small = 0

for genre in GENRES:
    genre_dir = os.path.join(DATA_ROOT, "genres_stems", genre)
    songs = os.listdir(genre_dir)

    for song in songs:
        song_dir = os.path.join(genre_dir, song)

        for stem in STEMS.values():
            file_path = os.path.join(song_dir, stem)
            if not os.path.isfile(file_path):
                continue

            size = os.path.getsize(file_path)

            if size > 5.0493 * 1024 * 1024:
                large += 1
            if size < 5.0491 * 1024 * 1024:
                small += 1

print("Q2:", abs(large - small))

Q2: 1072


In [28]:
# Q3

train_reggae_drums = len(tr["reggae"]["drums"])
val_country_vocals = len(val["country"]["vocals"])

print("Q3:", abs(train_reggae_drums - val_country_vocals))

Q3: 66


In [15]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    
    # ------------------- write your code here -------------------------------
    import librosa
    import numpy as np
    import pandas as pd
    
    total_files = 0

    for genre, stems in dataset_dict.items():
        for stem_name, file_list in stems.items():

            for file_path in file_list:
                total_files += 1
                try:
                    y, sr = librosa.load(file_path, sr=sr)
                    total_duration = len(y) / sr

                    # Detect non-silent intervals
                    intervals = librosa.effects.split(y, top_db=top_db)

                    # CASE A: fully silent file
                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type = ["full"]

                    else:
                        silence_segments = []
                        prev_end = 0

                        # Detect silence between intervals
                        for start, end in intervals:
                            if start > prev_end:
                                silence_segments.append((prev_end, start))
                            prev_end = end

                        # trailing silence
                        if prev_end < len(y):
                            silence_segments.append((prev_end, len(y)))

                        # Compute longest silence
                        max_silence = 0
                        silence_type = []

                        for start, end in silence_segments:
                            silence_len = (end - start) / sr

                            if silence_len > max_silence:
                                max_silence = silence_len

                                # Detect location
                                if start == 0:
                                    silence_type = ["start"]
                                elif end == len(y):
                                    silence_type = ["end"]
                                else:
                                    silence_type = ["middle"]

                    # Store result only if silence >= threshold
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })

                except Exception:
                    continue
    #-------------------------------------------------------------------------

    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------
# Pivot table: count of silent files by genre and stem
pivot_table = df_silence.pivot_table(index="Genre", columns="Stem", values="File_Path", aggfunc="count", fill_value=0)
print(pivot_table)
#-------------------------------------------------------------------------

Stem       bass  drums  other  vocals
Genre                                
blues        17     22      5      43
classical    68     57      5      69
country      16     16      2      16
disco         8      2      3      18
hiphop       20      3     22       6
jazz         25     24      1      70
metal         6      2      1      40
pop          10      5      2       4
reggae        5      4      7      12
rock         10      7      1      26


In [16]:
# Q4

len(df_silence)

680

In [17]:
# Q5 

df_silence[df_silence["Stem"] == "vocals"].shape[0]

304

In [18]:
# Q6

df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean()

np.float64(12.590789473684211)

In [19]:
# Q7

df_silence[(df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums")].shape[0]

24

In [20]:
# Q8

df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle")
].shape[0]

16

In [21]:
# Q9

df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0]

7

In [22]:
stems_audio = []
try:
    for key in STEM_KEYS:
        # Find the song folder path
        genre_dir = os.path.join(DATA_ROOT, "genres_stems", GENRE_TO_TEST)
        songs = sorted(os.listdir(genre_dir))
        song_dir = os.path.join(genre_dir, songs[SONG_INDEX])

        # Load the stem audio (limit to 5 sec for speed)
        stem_path = os.path.join(song_dir, STEMS[key])
        y, sr = librosa.load(stem_path, sr=None, duration=5.0)

        stems_audio.append(y)

    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
    #-------------------------------------------------------------------------

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [23]:
# ------------------- write your code here -------------------------------

# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#-------------------------------------------------------------------------

In [24]:
print("Mix Length:", len(mix_raw))
print("RMS:", round(rms_val, 2))
print("Peak normalized max:", round(np.max(np.abs(mix_norm)), 2))

Mix Length: 220500
RMS: 0.2
Peak normalized max: 1.0
